# Session2
## Feature Extracting From Textual Data:
In this notebook we will learn how to extract features from text data using different vectorizers, discussing their cons and pros and how have they evolved to solve the problems of its predecessors.

The vectorizers we're going to study:
 - **One-Hot-Encoding (OHE)**.
 - **Count Vectorizer (also known as: Bag Of Words (BOW))**.
 - **N-Grams(Bag-Of_N_Grams (BoN))**.
 - **Co-Occurrence Matrix**.
 - **Hash Vectorizing**.
    

## One-Hot-Encoding:
One-hot encoding works by representing each word in a document as a unique binary vector, where each word is mapped to a vector of zeros with a 1 in the position corresponding to that word's index in the vocabulary. This means that each word in the vocabulary will be represented by a unique binary vector, and the dimension of the vector will be equal to the size of the vocabulary.


To extract features using OHE we need:
 - Importing the `pandas` library, to support the DataFrame format.
 - `.split()`: method is used to split a `string` into a `list`. By default any whitespace is a separator.
 - `pd.get_dummies(data, dtype)`: Convert categorical variable into dummy/indicator variables. It returns a DataFrame where the names of the coloumns are the values of the `data`. `dtype=int`: to present the values as integers,otherwise, it will present them as boolean values.

In [1]:
Text = "I am learning NLP"
# Importing the library
import pandas as pd
# Generating the features
pd.get_dummies(Text.split(), dtype= int)

,I,NLP,am,learning
0,1,0,0,0
1,0,0,1,0
2,0,0,0,1
3,0,1,0,0


Although `pd.get_dummies()` is a good way for a quick analysis, but it has some very serious problemes -especially if used in ML pipeline- such as Column mismatching or Random cloumn ordering  which can lead the model to get crushed or getting incorrect predictions. So it's not advicable to use it in an NLP pipeline.

### Here's another example of how to use OHE , but in this case we're using numpy

In [2]:
text = "I love NLP and I will learn NLP in 2month"

words = text.split()

# Using a set to store the splitted words will automatically removes duplicates
vocabulary = sorted(set(words))

# Creates an empty list to store the final pairs of (word, vector).
one_hot_encoded = []

for word in vocabulary:

    #Creating a list of zeros. The length of this list is exactly equal to the number of unique words in our vocabulary   
    encoding = [0] * len(vocabulary)
    
    # Converts the set back into a list so we can find the specific position (index) of the current word.
    index = list(vocabulary).index(word)
    
    # Set the value at the index to 1 to indicate word presence
    encoding[index] = 1
    one_hot_encoded.append((word, encoding))

# Print the one-hot encoded vectors
for word, encoding in one_hot_encoded:
    print(f"{word}: {encoding}")

2month: [1, 0, 0, 0, 0, 0, 0, 0]
I: [0, 1, 0, 0, 0, 0, 0, 0]
NLP: [0, 0, 1, 0, 0, 0, 0, 0]
and: [0, 0, 0, 1, 0, 0, 0, 0]
in: [0, 0, 0, 0, 1, 0, 0, 0]
learn: [0, 0, 0, 0, 0, 1, 0, 0]
love: [0, 0, 0, 0, 0, 0, 1, 0]
will: [0, 0, 0, 0, 0, 0, 0, 1]


### Here's anotehr example (This time using the OneHotEncoder provided by sklearn):
`OneHotEncoder`: The industry-standard tool for creating dummy variables.

`string`: Used here specifically to get a list of punctuation marks (like `.`, `,`, `!`).

`sparse_output=False`: Forces the output to be a standard NumPy array (0s and 1s) that you can easily read, rather than a "sparse matrix" (a memory-saving format).

`np.array(vocabulary).reshape(-1, 1)`: Scikit-Learn expects a 2D array (like a table with rows and columns). This turns the list of words into a single column of many rows.

`fit_transform(...)`:
 - `fit`: The encoder "learns" the words in our vocabulary.
 - `transform`: It converts those words into their binary vector format.

In [3]:
# Importing needed libraries
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import string

# In this example we wil use multiple documents
documents = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "Cats and dogs are pets."
]

# Applying text preprocessing ( lowering, stripping , removing punctuation, and tokenization)
words = [word.lower().strip(string.punctuation)
         for doc in documents for word in doc.split()]

# Creating a set of vocabulries and ordering it alphabetically
vocabulary = sorted(set(words))

# Creating the encoder object
encoder = OneHotEncoder(sparse_output=False)

# Fitting and transforming the data
one_hot_vectors = encoder.fit_transform(np.array(vocabulary).reshape(-1, 1))

# Creating a dictionary comprehension
word_to_onehot = {vocabulary[i]: one_hot_vectors[i]
                  for i in range(len(vocabulary))}

for word, vector in word_to_onehot.items():
    print(f"Word: {word}, One-Hot Encoding: {vector}")

Word: and, One-Hot Encoding: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Word: are, One-Hot Encoding: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Word: cat, One-Hot Encoding: [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Word: cats, One-Hot Encoding: [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
Word: dog, One-Hot Encoding: [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
Word: dogs, One-Hot Encoding: [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
Word: log, One-Hot Encoding: [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
Word: mat, One-Hot Encoding: [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
Word: on, One-Hot Encoding: [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
Word: pets, One-Hot Encoding: [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
Word: sat, One-Hot Encoding: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
Word: the, One-Hot Encoding: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


### Pros and Cons of One-Hot-Encoding:
**Pros:**
 - Simplicity: One-hot encoding is easy to understand and implement. It’s a straightforward method to convert text to numerical format.
 - Interpretability: The vectors are easily interpretable and directly map to words.

**Cons:** 
 - High Dimensionality: The size of the one-hot vector is equal to the number of words in the vocabulary. For large corpora, this results in high-dimensional vectors, making computations more resource-intensive. For instance, if you have 100,000 unique words in your corpus, each word will be represented by a 100,000-dimensional vector, which    is computationally expensive.

 - Sparse Representation: Since the vectors are sparse (most values are 0), they do not capture any information about the relationships between words. For example, the words "king" and "queen" would be represented by completely different one-hot vectors, even though they have similar meanings.

 - Lack of Semantic Information: One-hot encoding does not capture any semantic meaning or context of words. For example, "dog" and "puppy" are different words but are semantically similar, and one-hot encoding treats them as completely unrelated.

 - Storage and Computation Efficiency: One-hot encoding can be inefficient in terms of storage and computation, especially for large datasets with extensive vocabularies.

## Count Vectorizing (Bag Of Words BOW):
Count Vectorizer is used to transform a given text into a vector **based on the frequency (count)** of each word that occurs in the entire text. This is helpful when we have multiple such texts and we wish to convert each word in each text into vectors (for using in further text analysis).

CountVectorizer creates a matrix in which each unique word is represented by a column of the matrix and each text sample from the document is a row in the matrix. The value of each cell is nothing but the count of the word in that particular text sample.






`from sklearn.feature_extraction.text import CountVectorizer`:Imports the `CountVectorizer` class. Unlike `OneHotEncoder`, this is specifically designed for text data.

`vectorizer.fit(text)`: The vectorizer scans the text and identifies all unique words. It assigns a specific index number to each word alphabetically.

`vectorizer.transform(text)`: It converts the sentence into a numerical vector based on the counts of the words learned in the `fit` step.The result is stored as a Sparse Matrix (to save memory).

`vector.toarray()`: Converts the sparse matrix into a readable NumPy array.

In [4]:
# Importing the CountVectorizer class 
from sklearn.feature_extraction.text import CountVectorizer

# Text
text = ["I love NLP and I will learn NLP in 2month "]

# Initializes the object. By default, it converts everything to lowercase and removes punctuation.
vectorizer = CountVectorizer()

# tokenizing
vectorizer.fit(text)

# encode document
vector = vectorizer.transform(text)

# summarize & generating output
print(vectorizer.vocabulary_) # Prints a dictionary showing which word belongs to which index.
# Note: "I" is usually ignored by default if it's considered a one-letter word depending on the version/settings of the vectorizer.

print(vector.toarray())

{'love': 4, 'nlp': 5, 'and': 1, 'will': 6, 'learn': 3, 'in': 2, '2month': 0}
[[1 1 1 1 1 2 1]]


#### Here's another example on Count Vectorizer using multiple documents:

In [5]:
documents = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "Cats and dogs are pets."
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

print(vectorizer.get_feature_names_out())
print("\n",X.toarray(), "\n")

['and' 'are' 'cat' 'cats' 'dog' 'dogs' 'log' 'mat' 'on' 'pets' 'sat' 'the']

 [[0 0 1 0 0 0 0 1 1 0 1 2]
 [0 0 0 0 1 0 1 0 1 0 1 2]
 [1 1 0 1 0 1 0 0 0 1 0 0]] 



### Pros and Cons of Count Vectorizer :
#### Pros:
1. Like one-hot encoding, BoW is fairly simple to understand and implement.
2. Fixed-length encoding for any sentence of arbitrary length.
3. Captures Word Frequency: It represents how often each word occurs in the document, which can be important in some tasks like topic modeling or spam detection.

#### Cons:
1. Sparsity: The BoW vectors tend to be sparse, especially when the vocabulary is large. For example, if the corpus has 10,000 words in its vocabulary, most vectors will have a lot of zero values, leading to inefficient memory usage.
2. Lack of Word Order: One of the biggest drawbacks is that the BoW model ignores the order of words. For example, "I love cats" and "Cats love I" would be represented in the same way, despite having different meanings.
3. No Semantic Information: BoW doesn’t capture semantic meaning or relationships between words. For example, synonyms like “car” and “automobile” would be treated as completely different words even though they have similar meanings.
4. High Dimensionality: For large vocabularies, BoW leads to high-dimensional vectors, which can be computationally expensive and lead to overfitting, especially when there are a lot of irrelevant words.
5. Difficulty with Rare Words: Words that appear very infrequently in the corpus (rare words) may be overly emphasized or irrelevant, which can distort the model.


### N-grams (Bag of N-Grams (BoN)):
All the representation schemes we’ve seen so far treat words as independent units. There is no notion of phrases or word order. The bag-of-n-grams (BoN) approach tries to remedy this. It does so by breaking text into chunks of n contiguous words (or tokens). This can help us capture some context, which earlier approaches could not do. Each chunk is called an n-gram.

**You can simply say that Count Vectorizer (Bag of words (BoW)) is a special case of the Bag of n-grams having n = 1.**
An N-Gram is a contiguous sequence of N items (usually words or characters) from a given text.
It helps to preserve some level of word order/context unlike Count Vectorizer or One-Hot_Encoder (which treat words independently).

#### Types of N-Grams:
 - Unigrams (1-grams): Single words, e.g., “cat,” “dog.”
 - Bigrams (2-grams): Pairs of consecutive words, e.g., “natural language,” “deep learning.”
 - Trigrams (3-grams): Triplets of consecutive words, e.g., “machine learning model,” “data science approach.”
 - 4-grams, 5-grams, etc.: Sequences of four, five, or more consecutive words.




- `from textblob import TextBlob`: imports the main class from the `textblob` library.
  
- *Note*: Unlike Scikit-Learn, TextBlob can process a raw string directly without wrapping it in a list.
 
- `TextBlob`: is a simplified library built on top of NLTK, designed for easy Natural Language Processing (NLP) tasks like sentiment analysis and noun phrase extraction.

- `TextBlob(Text)`:Converts the string into a TextBlob object so you can use its built-in NLP methods.
  
- `.ngrams(n=1)`: This method breaks the text into sequences of length \(n\).


In [6]:
Text = "I am learning NLP"

#Import textblob
from textblob import TextBlob

#For unigram : Use n = 1
TextBlob(Text).ngrams(1)

[WordList(['I']), WordList(['am']), WordList(['learning']), WordList(['NLP'])]

In [7]:
#For Bigram : For bigrams, use n = 2
TextBlob(Text).ngrams(2)

[WordList(['I', 'am']),
 WordList(['am', 'learning']),
 WordList(['learning', 'NLP'])]

- `ngram_range=(2,2)`:
   - The first `2` is the minimum size of the N-gram.
   - The second `2` is the maximum size.

- This forces the vectorizer to only extract **Bigrams** (pairs of two words).
  

In [8]:
#Bigram-based features for a document
#importing the function
from sklearn.feature_extraction.text import CountVectorizer

# Text
text = ["I love NLP and I will learn NLP in 2month "]

# create the transform

vectorizer = CountVectorizer(ngram_range=(2,2))
# tokenizing
vectorizer.fit(text)

# encode document
vector = vectorizer.transform(text)

# summarize & generating output
print(vectorizer.vocabulary_)
print(vector.toarray())

{'love nlp': 3, 'nlp and': 4, 'and will': 0, 'will learn': 6, 'learn nlp': 2, 'nlp in': 5, 'in 2month': 1}
[[1 1 1 1 1 1 1]]


### Pros and Cons of N-Grams:
#### Pros:
1. Captures context: Maintains some sequential word information (better than BoW).
2. Useful in text classification: Helps in tasks like sentiment analysis, spam detection.
3. Flexible: You can control how much context you want (by adjusting N).

#### Cons:
1. High dimensionality: As N increases, vocabulary size increases rapidly.
2. Data sparsity: Many rare combinations → sparse matrix. With larger N-grams it becomes less likely to find repeated instances of the same sequence leading to sparse data.
3. No semantic meaning: Like BoW, it treats phrases as symbols, not meanings. While N-grams are good at recognizing patterns they lack the understanding of context beyond the sequences they were trained on.
4. Fails on unseen N-grams: Does not generalize well to new word sequences.
5. Lack of long-range context: They only consider nearby words and ignore broader sentence meaning.

### Co-Occurrence Matrix:
A co-occurrence matrix is a mathematical representation that captures the frequency with which pairs of words appear together within a specified context, such as a sentence, paragraph, or document. It is a square matrix where rows and columns represent unique words in the corpus, and each cell (i, j) contains the number of times word i appears in the context of word j.

Given a vocabulary of N unique words, a co-occurrence matrix C is an N x N matrix, where:
C\[i]\[j] = the number of times word j appears in the context of word i.


- `vocab_to_index`: Creates a dictionary where each word maps to a number (e.g., `'nlp': 0`, `'love': 1`). This acts as the **coordinates** for the matrix.
- `bigrams()`: A utility from `nltk` that creates sliding pairs of words.
- `bigram_freq = nltk.FreqDist(bi_grams).most_common()`: `FreqDist`: Counts how many times each pair appears.
- `co_occurrence_matrix = np.zeros((len(vocab), len(vocab)))`: Creates a square grid (Matrix) of zeros.The size is` Vocabulary Size x Vocabulary` Size. Every word has its own row and its own column.
- `CoMatrixFinal = pd.DataFrame(matrix, index=vocab_to_index.keys(), columns=vocab_to_index.keys())`: Converts the raw NumPy numbers into a labeled table.The Index (Rows) and Columns are labeled with the words from your vocabulary, making it easy to read which word co-occurs with which.
- 


In [9]:
import numpy as np
import nltk
from nltk import bigrams
import itertools
import pandas as pd  # Added for DataFrame support

def co_occurrence_matrix(corpus):

    # Create vocabulary set and convert to list to fix order
    vocab = set(corpus)
    vocab = list(vocab)  # Convert to list for consistent ordering
    
    # Create word to index mapping
    vocab_to_index = {word:i for i, word in enumerate(vocab)}
    print("vocab to index")
    print(vocab_to_index)
    print()
    
    # Generate all bigrams from the corpus
    bi_grams = list(bigrams(corpus))
    print(bi_grams)
    print()
    
    # Calculate frequency distribution of bigrams
    # Returns list of tuples: ((word1, word2), count)
    bigram_freq = nltk.FreqDist(bi_grams).most_common()
    print(bigram_freq)
    print()
    
    # Initialize co-occurrence matrix with zeros
    # Dimensions: vocab_size x vocab_size
    co_occurrence_matrix = np.zeros((len(vocab), len(vocab)))
    
    # Populate the co-occurrence matrix
    for bigram in bigram_freq:
        current = bigram[0][1]   # Second word in bigram
        previous = bigram[0][0]  # First word in bigram
        count = bigram[1]        # Occurrence count
        
        # Get matrix positions
        pos_current = vocab_to_index[current]
        pos_previous = vocab_to_index[previous]
        
        # Update matrix (current word as row, previous word as column)
        co_occurrence_matrix[pos_current][pos_previous] = count
    
    # Convert to matrix type for better handling
    co_occurrence_matrix = np.matrix(co_occurrence_matrix)
    
    return co_occurrence_matrix, vocab_to_index

# Example usage
sentences = [['I', 'love', 'nlp'],
             ['I', 'love', 'to', 'learn'],  # Fixed missing comma
             ['nlp', 'is', 'future'],
             ['nlp', 'is', 'cool']]

# Flatten the list of sentences into single corpus list
merged = list(itertools.chain.from_iterable(sentences))
print(merged)
print()

# Generate co-occurrence matrix
matrix, vocab_to_index = co_occurrence_matrix(merged)

# Convert to pandas DataFrame for better visualization
CoMatrixFinal = pd.DataFrame(matrix, 
                            index=vocab_to_index.keys(),  # Row labels
                            columns=vocab_to_index.keys())  # Column labels

# Display the matrix
print("Co-occurrence Matrix:")
print(CoMatrixFinal)

['I', 'love', 'nlp', 'I', 'love', 'to', 'learn', 'nlp', 'is', 'future', 'nlp', 'is', 'cool']

vocab to index
{'cool': 0, 'to': 1, 'is': 2, 'I': 3, 'future': 4, 'love': 5, 'learn': 6, 'nlp': 7}

[('I', 'love'), ('love', 'nlp'), ('nlp', 'I'), ('I', 'love'), ('love', 'to'), ('to', 'learn'), ('learn', 'nlp'), ('nlp', 'is'), ('is', 'future'), ('future', 'nlp'), ('nlp', 'is'), ('is', 'cool')]

[(('I', 'love'), 2), (('nlp', 'is'), 2), (('love', 'nlp'), 1), (('nlp', 'I'), 1), (('love', 'to'), 1), (('to', 'learn'), 1), (('learn', 'nlp'), 1), (('is', 'future'), 1), (('future', 'nlp'), 1), (('is', 'cool'), 1)]

Co-occurrence Matrix:
        cool   to   is    I  future  love  learn  nlp
cool     0.0  0.0  1.0  0.0     0.0   0.0    0.0  0.0
to       0.0  0.0  0.0  0.0     0.0   1.0    0.0  0.0
is       0.0  0.0  0.0  0.0     0.0   0.0    0.0  2.0
I        0.0  0.0  0.0  0.0     0.0   0.0    0.0  1.0
future   0.0  0.0  1.0  0.0     0.0   0.0    0.0  0.0
love     0.0  0.0  0.0  2.0     0.0   0.0    0

### Pros and Cons of Co-Occurrence Matrix Vectorizer :
#### Pros:
- **Semantic Relationships:** The co-occurrence matrix helps capture semantic relationships between words. Words that frequently appear together are likely to have related meanings or be used in similar contexts.

### Cons:
1. **Sparse Matrices:** Co-occurrence matrices are often sparse, meaning many cells contain zeros, especially for large vocabularies. Efficient storage and processing techniques, such as sparse matrix representations, are essential.
2. **Scalability:** For large corpora, constructing and manipulating co-occurrence matrices can be computationally intensive. Optimizations and parallel processing techniques are often necessary.

### Hash Vectorizer:
hashing vectorizer is a vectorizer which uses the *hashing Trick* to find the token string name to feature integer index mapping. Conversion of text documents into matrix is done by this vectorizer where it turns the collection of documents into a sparse matrix which are holding the token occurence counts.

- `vector = vectorizer.transform(text)`: Converts the words into the 10-dimensional space.
- *Note*: **No** fit step: Notice there is **no** `vectorizer.fit(text)`. Hashing doesn't need to "learn" the words; it just calculates them on the fly.

In [10]:
from sklearn.feature_extraction.text import HashingVectorizer

# list of text documents

text = ["The quick brown fox jumped over the lazy dog."]
# transform
vectorizer = HashingVectorizer(n_features=10)

# create the hashing vector
vector = vectorizer.transform(text)

# summarize the vector
print(vector.shape)
print(vector.toarray())

(1, 10)
[[ 0.          0.57735027  0.          0.          0.          0.
   0.         -0.57735027 -0.57735027  0.        ]]


#### Another Example to Hash Vecotrizer:

In [11]:
Sample_text = ["Jon is playing football.",
               "He loves to play football.",
               "He is just 10 years old.",
               "His favorite player is Cristiano Ronaldo."]

vectorizer = HashingVectorizer(n_features=2**4)

vector = vectorizer.transform(Sample_text)

print(vector.toarray(),"\n")

print(vector.shape)


[[ 0.          0.5         0.          0.          0.          0.
   0.          0.          0.          0.          0.5         0.
   0.          0.5         0.         -0.5       ]
 [ 0.          0.          0.          0.57735027  0.          0.
   0.          0.57735027  0.          0.          0.          0.57735027
   0.          0.          0.          0.        ]
 [ 0.         -0.40824829  0.          0.40824829 -0.40824829 -0.40824829
   0.          0.         -0.40824829  0.          0.          0.
   0.          0.40824829  0.          0.        ]
 [ 0.5         0.         -0.5         0.          0.          0.
   0.          0.          0.         -0.5         0.         -0.5
   0.          0.          0.          0.        ]] 

(4, 16)


### Pros of using Hash Vecrorizer:
**Pros:**
1. **Low Memory Usage:** There is no need of storing the vocabulary dictionary in the memory, for large data sets it is very low memory scalable, This is ideal for millions of documents where a CountVectorizer would crash your RAM.
2. **Streaming Friendly:** We can process data in small batches (Online Learning) because the mapping of a word to an index never changes. As there in no state during the fit, it can be used in a streaming or parallel pipeline.
3. **Fixed Output Size:** We decide exactly how many columns (features) the output has from the start (e.g., `n_features=10000`). 

**Cons**:
1. **Feature Collisions:** Two different words might hash to the same index (e.g., "apple" and "orange" might both end up in bucket #45). This can affect the accuracy of the model.
2. **One-Way Transformation (No Inverse):**  We **cannot** "un-hash" a vector; Once a word is turned into a number, we *can't* ask the vectorizer, "What word does index 452 represent?".
3. **Lack of interpretability:** Since we can't see the words associated with the features, it is difficult to explain why our model made a certain prediction.